# DDInter 2.0 - Drug-Food Interaction Scraper
**Muc tieu:** Thu thap du lieu tuong tac thuoc-thuc pham tu DDInter 2.0

> Chay tung cell theo thu tu tu tren xuong duoi.

In [ ]:
# Cell 1: Cai thu vien neu chua co
# !pip install requests

In [ ]:
# Cell 2: Import thu vien
import time
import csv
import re
import requests

print("Import thanh cong!")

In [ ]:
# Cell 3: Cau hinh
BASE_URL    = "https://ddinter2.scbdd.com"
PAGE_URL    = BASE_URL + "/server/other_interaction/"
DATA_URL    = BASE_URL + "/server/food-interaction-source/"
OUTPUT_FILE = "ddinter2_dfi_data.csv"  # File se luu cung thu muc voi file .ipynb nay
PAGE_SIZE   = 10

SEVERITY_MAP = {"1": "Minor", "2": "Moderate", "3": "Major"}

HEADERS = {
    "User-Agent":         "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36",
    "Accept-Language":    "vi,vi-VN;q=0.9,fr-FR;q=0.8,fr;q=0.7,en-US;q=0.6,en;q=0.5,zh-CN;q=0.4,zh;q=0.3",
    "Connection":         "keep-alive",
    "sec-ch-ua":          '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    "sec-ch-ua-mobile":   "?0",
    "sec-ch-ua-platform": '"Windows"',
}

print("Cau hinh xong!")

In [ ]:
# Cell 4: Ham lay CSRF token
# Buoc 1: Truy cap trang de server cap cookie
# Buoc 2: Doc cookie do ra lay token
# Neu khong co cookie -> dung token gia (fallback)
def get_csrf_token(session):
    print("Truy cap trang de lay session...")
    session.get(
        PAGE_URL,
        headers={**HEADERS, "Accept": "text/html,application/xhtml+xml,*/*;q=0.9"},
        timeout=20
    )
    token = session.cookies.get("csrftoken", "")
    if token:
        print(f"Token tu cookie: {token[:20]}...")
        return token
    print("Khong tim duoc token -- thu gui POST voi token gia")
    return "dummy_token_not_enforced"

print("Dinh nghia ham get_csrf_token xong!")

In [ ]:
# Cell 5: Ham dong goi du lieu gui len server (phan trang)
# start: bat dau tu ban ghi thu may
# length: lay bao nhieu ban ghi moi lan
def build_post_data(csrf_token, start, length):
    data = {
        "csrfmiddlewaretoken": csrf_token,
        "draw":             str((start // length) + 1),
        "start":            str(start),
        "length":           str(length),
        "search[value]":    "",
        "search[regex]":    "false",
        "severity":         "",
        "mechanism":        "",
    }
    for i in range(8):
        data[f"columns[{i}][data]"]          = "null"
        data[f"columns[{i}][name]"]          = ""
        data[f"columns[{i}][searchable]"]    = "true"
        data[f"columns[{i}][orderable]"]     = "false"
        data[f"columns[{i}][search][value]"] = ""
        data[f"columns[{i}][search][regex]"] = "false"
    return data

print("Dinh nghia ham build_post_data xong!")

In [ ]:
# Cell 6: Ham test ket noi bang cach lay 5 ban ghi dau tien
# Chay truoc de dam bao moi thu hoat dong truoc khi tai toan bo
def test_connection(session, csrf_token):
    print("Kiem tra ket noi (lay 5 ban ghi dau)...")
    post_headers = {
        **HEADERS,
        "Accept":           "application/json, text/javascript, */*; q=0.01",
        "Content-Type":     "application/x-www-form-urlencoded; charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Origin":           BASE_URL,
        "Referer":          PAGE_URL,
        "Sec-Fetch-Dest":   "empty",
        "Sec-Fetch-Mode":   "cors",
        "Sec-Fetch-Site":   "same-origin",
    }
    data = build_post_data(csrf_token, 0, 5)
    r = session.post(DATA_URL, headers=post_headers, data=data, timeout=20)
    print(f"HTTP: {r.status_code}, Content-Type: {r.headers.get('Content-Type','')}")

    if r.status_code != 200:
        print(f"Loi: {r.text[:300]}")
        return False

    try:
        resp = r.json()
        total = resp.get("recordsTotal", 0)
        rows  = resp.get("data", [])
        print(f"Thanh cong! Tong: {total} ban ghi, nhan duoc: {len(rows)} ban ghi")
        if rows:
            print(f"Vi du: {rows[0].get('drugName','')} + {rows[0].get('foodName','')}")
        return True
    except Exception as e:
        print(f"Khong parse duoc JSON: {e}")
        print(f"Response: {r.text[:500]}")
        return False

print("Dinh nghia ham test_connection xong!")

In [ ]:
# Cell 7: Ham tai toan bo du lieu theo tung trang
# Moi request nghi 0.5 giay de tranh bi server chan
def fetch_all(session, csrf_token, page_size=200):
    print(f"Tai toan bo du lieu (page_size={page_size})...")
    all_records = []
    start = 0
    total = None

    post_headers = {
        **HEADERS,
        "Accept":           "application/json, text/javascript, */*; q=0.01",
        "Content-Type":     "application/x-www-form-urlencoded; charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Origin":           BASE_URL,
        "Referer":          PAGE_URL,
        "Sec-Fetch-Dest":   "empty",
        "Sec-Fetch-Mode":   "cors",
        "Sec-Fetch-Site":   "same-origin",
    }

    while True:
        data = build_post_data(csrf_token, start, page_size)
        try:
            r = session.post(DATA_URL, headers=post_headers, data=data, timeout=20)

            if r.status_code != 200:
                print(f"HTTP {r.status_code}: {r.text[:200]}")
                break

            resp = r.json()

            if total is None:
                total = int(resp.get("recordsTotal", 0))
                print(f"Tong: {total} ban ghi")

            rows = resp.get("data", [])
            if not rows:
                break

            for row in rows:
                level_raw = str(row.get("level", ""))
                all_records.append({
                    "drug_name":   row.get("drugName",       ""),
                    "food_name":   row.get("foodName",       ""),
                    "severity":    SEVERITY_MAP.get(level_raw, level_raw),
                    "description": row.get("newInteraction", ""),
                    "source":      row.get("references",     ""),
                })

            print(f"{len(all_records)}/{total}")

            if len(all_records) >= total:
                break

            start += page_size
            time.sleep(0.5)

        except Exception as e:
            print(f"Loi tai start={start}: {e}")
            break

    return all_records

print("Dinh nghia ham fetch_all xong!")

In [ ]:
# Cell 8: Ham luu du lieu ra file CSV
# Dung encoding utf-8-sig de Excel doc duoc tieng Viet
def save_csv(records):
    fields = ["drug_name", "food_name", "severity", "description", "source"]
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(records)
    print(f"Da luu {len(records)} ban ghi vao {OUTPUT_FILE}")

print("Dinh nghia ham save_csv xong!")

In [ ]:
# Cell 9: CHAY CHUONG TRINH CHINH
# Goi lan luot: lay token -> test ket noi -> tai du lieu -> luu CSV
print("=" * 55)
print("  DDInter 2.0 - Drug-Food Interaction Scraper v2")
print("=" * 55)

session = requests.Session()
csrf_token = get_csrf_token(session)

ok = test_connection(session, csrf_token)
if not ok:
    print("Ket noi that bai. Dung lai.")
else:
    records = fetch_all(session, csrf_token, page_size=200)

    if records:
        save_csv(records)
        print("Preview 3 ban ghi dau:")
        for r in records[:3]:
            print(f"  {r['drug_name']} + {r['food_name']} -> {r['severity']}")
    else:
        print("Khong lay duoc du lieu.")

# File CSV se duoc luu cung thu muc voi file .ipynb nay
import os
print(f"\nDuong dan file: {os.path.abspath(OUTPUT_FILE)}")